# Check the results of the Desc metadetect shear matching

In [ ]:
# Standard imports
import matplotlib.pyplot as plt
import numpy as np
import tables_io
import hpmcm

## Configuration

Set the tract number and define input file paths.

In [ ]:
tract = 8006

# Load shear catalogs and match output tables
shear_desc_in = tables_io.read(f'{tract}/mdet-run-{tract}.parq')
shear_desc_ns = tables_io.read(f'{tract}/mdet-run-reduced_uncleaned_{tract}_ns.parq')
shear_desc_1m = tables_io.read(f'{tract}/mdet-run-reduced_uncleaned_{tract}_1m.parq')
shear_desc_1p = tables_io.read(f'{tract}/mdet-run-reduced_uncleaned_{tract}_1p.parq')
shear_desc_obj_stats = tables_io.read(f'{tract}/desc_md_match_object_stats.pq')
shear_desc_obj_assoc = tables_io.read(f'{tract}/desc_md_match_object_assoc.pq')
input_files = [
    f'{tract}/mdet-run-reduced_uncleaned_{tract}_ns.parq',
    f'{tract}/mdet-run-reduced_uncleaned_{tract}_1m.parq',
    f'{tract}/mdet-run-reduced_uncleaned_{tract}_1p.parq',
]    
catalog_ids = [0, 1, 2]

## Validation plots — positional checks

In [ ]:
_ = plt.scatter(shear_desc_obj_stats.x_pix, shear_desc_obj_stats.y_pix, s=1)

In [ ]:
_ = plt.scatter(shear_desc_ns.x_pix, shear_desc_ns.y_pix, s=1)

In [ ]:
# Join source catalog data onto the object stats table
joined_shear_obj_full = hpmcm.output_tables.buildJoinedObjectTable(
    shear_desc_obj_stats,
    shear_desc_obj_assoc,
    input_files,
    catalog_ids,
)

In [ ]:
center_mask = np.bitwise_and(np.fabs(joined_shear_obj_full.x_cent - 75) < 75, np.fabs(joined_shear_obj_full.y_cent - 75) < 75)


In [ ]:
joined_shear_obj = joined_shear_obj_full[center_mask]
joined_shear_obj_edge = joined_shear_obj_full[~center_mask]

In [ ]:
_ = plt.scatter(joined_shear_obj.y_cell_coadd_0, 3600*(joined_shear_obj.ra - joined_shear_obj.ra_0), s=1)

In [ ]:
_ = plt.scatter(joined_shear_obj_edge.x_cent, joined_shear_obj_edge.y_cent, s=1)

In [ ]:
joined_shear_obj.x_cent

In [ ]:
_ = plt.scatter(joined_shear_obj.x_cell_coadd_0, joined_shear_obj.x_pix - joined_shear_obj.x_pix_1, s=1)
_ = plt.scatter(joined_shear_obj.x_cell_coadd_0, joined_shear_obj.x_pix - joined_shear_obj.x_pix_2, s=1)

In [ ]:
_ = plt.scatter(joined_shear_obj.x_cell_coadd_0, 3600*(joined_shear_obj.ra - joined_shear_obj.ra_1), s=1)
_ = plt.scatter(joined_shear_obj.x_cell_coadd_0, 3600*(joined_shear_obj.ra - joined_shear_obj.ra_2), s=1)

In [ ]:
cut = np.bitwise_and(joined_shear_obj.n_sib_objects==1, joined_shear_obj.n_sib_srcs>3)
print(cut.sum())

In [ ]:
good_mask = joined_shear_obj.catalog_mask == 7
bad_mask = joined_shear_obj.catalog_mask != 7
good_sub = joined_shear_obj[good_mask]
bad_sub = joined_shear_obj[bad_mask]
missing_ns_mask = np.bitwise_and(joined_shear_obj.catalog_mask, 1) != 1
missing_1m_mask = np.bitwise_and(joined_shear_obj.catalog_mask, 2) != 2
missing_1p_mask = np.bitwise_and(joined_shear_obj.catalog_mask, 4) != 4
missing_ns = joined_shear_obj[missing_ns_mask]
missing_1m = joined_shear_obj[missing_1m_mask]
missing_1p = joined_shear_obj[missing_1p_mask]

In [ ]:
print('good', np.bitwise_and(joined_shear_obj.n_src==3, joined_shear_obj.n_unique==3).sum())
print('conf', (joined_shear_obj.n_src != joined_shear_obj.n_unique).sum())
print('missing 1', np.bitwise_and(joined_shear_obj.n_src==2, joined_shear_obj.n_unique==2).sum())
print('missing 2', np.bitwise_and(joined_shear_obj.n_src==1, joined_shear_obj.n_unique==1).sum())
print('split', np.bitwise_and(joined_shear_obj.n_sib_objects>1, joined_shear_obj.n_sib_srcs==3).sum())
print('weird', np.bitwise_and(joined_shear_obj.n_sib_objects>1, joined_shear_obj.n_sib_srcs>3).sum())
print('weird', np.bitwise_and(joined_shear_obj.n_sib_objects>1, joined_shear_obj.n_sib_srcs<3).sum())


In [ ]:
_ = plt.scatter(missing_ns.ra, missing_ns.dec, s=1)
_ = plt.scatter(missing_1m.ra, missing_1m.dec, s=1)
_ = plt.scatter(missing_1p.ra, missing_1p.dec, s=1)

In [ ]:
_ = plt.hist(good_sub.n_sib_srcs, bins=np.linspace(0.5, 9.5, 10), alpha=0.5)
_ = plt.hist(bad_sub.n_sib_srcs, bins=np.linspace(0.5, 9.5, 10), alpha=0.5)

_ = plt.yscale('log')

In [ ]:
_ = plt.hist(missing_ns.x_cent, bins=np.linspace(-5, 155, 16))
_ = plt.hist(missing_1m.x_cent, bins=np.linspace(-5, 155, 16))
_ = plt.hist(missing_1p.x_cent, bins=np.linspace(-5, 155, 16))

In [ ]:
_ = plt.scatter(good_sub.ra, good_sub.dec, s=1)
_ = plt.scatter(bad_sub.ra, bad_sub.dec, s=1)

In [ ]:
_ = plt.scatter(good_sub.x_cent[::5], good_sub.y_cent[::5], s=1)
_ = plt.scatter(bad_sub.x_cent[::5], bad_sub.y_cent[::5], s=1)

In [ ]:
_ = plt.hist2d(3600*(good_sub.ra-good_sub.ra_0), 3600*(good_sub.dec-good_sub.dec_0),
               bins=(np.linspace(-0.4, 0.4, 81), np.linspace(-0.4, 0.4, 81)), norm='log')

In [ ]:
_ = plt.hist(good_sub.dist_rms, bins=np.linspace(0, 0.4, 201))
_ = plt.yscale('log')

In [ ]:
dist_0 = np.sqrt(good_sub.x_pix - good_sub.x_pix_0)**2 + (good_sub.y_pix - good_sub.y_pix_0)**2
dist_1 = np.sqrt(good_sub.x_pix - good_sub.x_pix_1)**2 + (good_sub.y_pix - good_sub.y_pix_1)**2
dist_2 = np.sqrt(good_sub.x_pix - good_sub.x_pix_2)**2 + (good_sub.y_pix - good_sub.y_pix_2)**2

dist_0_b = np.sqrt(bad_sub.x_pix - bad_sub.x_pix_0)**2 + (bad_sub.y_pix - bad_sub.y_pix_0)**2
dist_1_b = np.sqrt(bad_sub.x_pix - bad_sub.x_pix_1)**2 + (bad_sub.y_pix - bad_sub.y_pix_1)**2
dist_2_b = np.sqrt(bad_sub.x_pix - bad_sub.x_pix_2)**2 + (bad_sub.y_pix - bad_sub.y_pix_2)**2

In [ ]:
_ = plt.scatter(good_sub.y_cent, good_sub.x_pix - good_sub.x_pix_0, s=1)
_ = plt.scatter(good_sub.y_cent, good_sub.x_pix - good_sub.x_pix_1, s=1)
_ = plt.scatter(good_sub.y_cent, good_sub.x_pix - good_sub.x_pix_2, s=1)

In [ ]:
_ = plt.scatter(good_sub.x_cent, good_sub.x_pix - good_sub.x_pix_0, s=1)
_ = plt.scatter(good_sub.x_cent, good_sub.x_pix - good_sub.x_pix_1, s=1)
_ = plt.scatter(good_sub.x_cent, good_sub.x_pix - good_sub.x_pix_2, s=1)

In [ ]:
#_ = plt.hist2d(good_sub.x_cent, good_sub.x_pix - good_sub.x_pix_0, bins=(np.linspace(0, 150, 151), np.linspace(-2, 2, 101)), norm='log')
_ = plt.hist2d(good_sub.x_cent, good_sub.x_pix_0 - good_sub.x_pix_1, bins=(np.linspace(0, 150, 151), np.linspace(-2, 2, 101)), norm='log')
#_ = plt.hist2d(good_sub.x_cent, good_sub.x_pix_1 - good_sub.x_pix_2, bins=(np.linspace(0, 150, 151), np.linspace(-2, 2, 101)), norm='log')

In [ ]:
_ = plt.scatter(good_sub.x_cent, good_sub.y_pix - good_sub.y_pix_0, s=1)
_ = plt.scatter(good_sub.x_cent, good_sub.y_pix - good_sub.y_pix_1, s=1)
_ = plt.scatter(good_sub.x_cent, good_sub.y_pix - good_sub.y_pix_2, s=1)

In [ ]:
_ = plt.hist(dist_0, bins=np.linspace(0, 1.4, 201), alpha=0.2)
_ = plt.hist(dist_1, bins=np.linspace(0, 1.4, 201), alpha=0.2)
_ = plt.hist(dist_2, bins=np.linspace(0, 1.4, 201), alpha=0.2)
_ = plt.yscale('log')

In [ ]:
_ = plt.hist(dist_0_b, bins=np.linspace(0, 1.4, 201), alpha=0.2)
_ = plt.hist(dist_1_b, bins=np.linspace(0, 1.4, 201), alpha=0.2)
_ = plt.hist(dist_2_b, bins=np.linspace(0, 1.4, 201), alpha=0.2)
_ = plt.yscale('log')

In [ ]:
_ = plt.hist2d(3600*(bad_sub.ra-bad_sub.ra_0), 3600*(bad_sub.dec-bad_sub.dec_0),
               bins=(np.linspace(-0.4, 0.4, 81), np.linspace(-0.4, 0.4, 81)), norm='log')

In [ ]:
_ = plt.hist(shear_desc_ns.x_cell_coadd, bins=np.linspace(-5, 155, 16))
_ = plt.hist(shear_desc_1m.x_cell_coadd, bins=np.linspace(-5, 155, 16))
_ = plt.hist(shear_desc_1p.x_cell_coadd, bins=np.linspace(-5, 155, 16))

In [ ]:
_ = plt.hist(joined_shear_obj.x_cent, bins=np.linspace(-5, 155, 161))
_ = plt.hist(good_sub.x_cent, bins=np.linspace(-5, 155, 161))
_ = plt.hist(bad_sub.x_cent, bins=np.linspace(-5, 155, 161))

In [ ]:
_ = plt.hist(good_sub.y_cent, bins=np.linspace(-5, 155, 161))
_ = plt.hist(bad_sub.y_cent, bins=np.linspace(-5, 155, 161))

In [ ]:
_ = plt.hist(joined_shear_obj.catalog_mask, bins=np.linspace(0.5, 7.5, 8), alpha=0.2)
_ = plt.hist(joined_shear_obj_edge.catalog_mask, bins=np.linspace(0.5, 7.5, 8), alpha=0.2)
_ = plt.yscale('log')